# DAY 17
# DATE - 05.06.2026
# Deep Learning for Remote Sensing — CNN Basics

# 1. CNN vs Random Forest (Spatial Context)
Point: CNN naturally spatial pixels ke "neighbor" relationship ko preserve karta hai.

Satellite Use-case: Satellite images mein objects (roads, crops, rivers) unke texture aur shape se pehchane jate hain. Random Forest pixel ko akela (isolated) dekhta hai, isliye shapes ko classify nahi kar pata.

In [20]:
import torch
import torch.nn as nn

# --- STEP 1: DEFINE ARCHITECTURE ---


In [21]:
class SatelliteCNN(nn.Module):
    # Yahan dhyan dena: num_classes ko bracket ke andar hona chahiye
    def __init__(self, num_classes=4):
        super(SatelliteCNN, self).__init__()

        # Conv Layer 1: Input=13 channels (Sentinel-2)
        self.conv1 = nn.Conv2d(in_channels=13, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()

        # Max Pooling: Size 64x64 -> 32x32
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv Layer 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()

        # Fully Connected Layer: 64 filters * 32 * 32 pixels
        self.fc = nn.Linear(64 * 32 * 32, num_classes)

    def forward(self, x):
        x = self.pool(self.relu1(self.conv1(x)))
        x = self.relu2(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# --- STEP 2: INITIALIZE & SUMMARY ---


In [22]:
model = SatelliteCNN(num_classes=4)

# Dummy Data: (Batch=1, Channels=13, Height=64, Width=64)
dummy_input = torch.randn(1, 13, 64, 64)
output = model(dummy_input)

# Parameter Count Function
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*30)
print("SATELLITE CNN SUMMARY")
print("="*30)
print(f"Input Shape:  {list(dummy_input.shape)}")
print(f"Output Shape: {list(output.shape)} (4 Classes)")
print(f"Total Trainable Parameters: {count_parameters(model):,}")
print("="*30)

SATELLITE CNN SUMMARY
Input Shape:  [1, 13, 64, 64]
Output Shape: [1, 4] (4 Classes)
Total Trainable Parameters: 284,420


# --- STEP 1: RGB CONFIGURATION (3 Channels) ---


In [23]:
# Hum wahi architecture use karenge bas input channel change kar denge

class FlexibleCNN(nn.Module):
    def __init__(self, in_channels, num_classes=4):
        super(FlexibleCNN, self).__init__()
        # Conv Layer: filters=16, kernel=3x3, padding=1
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Fully Connected Layer: 16 filters * 32 height * 32 width
        self.fc = nn.Linear(16 * 32 * 32, num_classes)

    # Dhyan se dekho: 'forward' function thik '__init__' ke niche sahi alignment me hai
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)  # Flattening the tensor
        x = self.fc(x)
        return x

In [24]:
print("--- RUNNING RGB EXPERIMENT (3 CHANNELS) ---")
rgb_model = FlexibleCNN(in_channels = 3)
rgb_input = torch.randn(1, 3, 64, 64)
rgb_out = rgb_model(rgb_input)
print(f"RGB Input {rgb_input.shape} -> Success! Output: {rgb_out.shape}")

print("\n--- SWITCHING BACK TO SATELLITE (13 CHANNELS) ---")
sat_model = FlexibleCNN(in_channels = 13)
sat_input = torch.randn(1, 13, 64, 64)
sat_out = sat_model(sat_input)
print(f"Satellite Input {sat_input.shape} -> Success! Output: {sat_out.shape}")

--- RUNNING RGB EXPERIMENT (3 CHANNELS) ---
RGB Input torch.Size([1, 3, 64, 64]) -> Success! Output: torch.Size([1, 4])

--- SWITCHING BACK TO SATELLITE (13 CHANNELS) ---
Satellite Input torch.Size([1, 13, 64, 64]) -> Success! Output: torch.Size([1, 4])


The Channel Secret (Input Shape)
Hamara input format hota hai: (Batch, Channels, Height, Width).

Normal AI models ke liye Channels = 3 (Red, Green, Blue) hota hai.

Satellite AI ke liye Channels = 13 hota hai (Sentinel-2).

Fayda: Isme NIR (Near Infrared) aur SWIR (Short-wave Infrared) jaise bands hote hain, jo badal (clouds), khet ki mitti ka moisture, aur jangal ki health ko aam aankhon se 10 guna behtar pehchante hain.

The Layer Math (Where do parameters come from?)
Hamare model me total 284,420 parameters (weights) hain.

Convolution Layers: Inka kaam sirf dacha aur patterns (edges, shapes) dhoondna hai, isme kam parameters kharch hote hain (approx 22,000).

Fully Connected (Linear) Layer: Yeh sabse aakhri layer hoti hai jo saare patterns ko aapas me jodkar final decision leti hai. Model ke 92% parameters (262,148) akele isi layer me hote hain.